# UR3 LLM Control Workspace

Đây là notebook tổng quan và hướng dẫn hoạt động của workspace ROS 2 điều khiển UR3 bằng LLM, Camera, MoveIt 2 và Gazebo.

> Nội dung được viết bằng tiếng Việt và dựa đúng trên cấu trúc hiện tại của workspace.

## 1. Tổng quan dự án

Hệ thống có pipeline:

```text
Câu lệnh tự nhiên
        ↓
9Router / LLM Planner
        ↓
Plan Validator
        ↓
Robot Skills C++
        ↓
MoveIt 2 + Gazebo gripper
```

Workspace hiện gồm hai package chính:

- `ur3_llm_control`: camera detector, planner LLM và validator kế hoạch.
- `ur3_llm_robot_skills`: executor skill, MoveIt configuration và mô phỏng UR3.

## 2. Kiến trúc hệ thống

```text
Camera Gazebo
    │
    ▼
/overhead_cam
    │
    ▼
camera_detector
    │
    ▼
/scene_state
    │
    ▼
ur3_llm_planner
    │
    ▼
9Router
    │
    ▼
/llm_task/validated_plan
    │
    ▼
ur3_skill_executor
    │
    ▼
MoveIt 2 + Gazebo UR3e
```

### Các bước chính

1. Camera phát hiện vật thể và zone.
2. `camera_detector` publish trạng thái vào `/scene_state`.
3. `ur3_llm_planner` gửi trạng thái cho 9Router và nhận kế hoạch JSON.
4. `task_validator` kiểm tra kế hoạch trước khi thực thi.
5. `ur3_skill_executor` thực thi `pick`, `place` và `home` qua MoveIt 2.

## 3. Cấu trúc workspace

```text
ur3_ws/
├── src/
│   ├── ur3_llm_control/
│   │   ├── config/
│   │   │   ├── 9router.env
│   │   │   └── student_config.yaml
│   │   ├── scripts/
│   │   │   ├── run_llm_robot.sh
│   │   │   ├── send_demo.sh
│   │   │   ├── send_demo_advanced.sh
│   │   │   ├── send_demo_conflict.sh
│   │   │   ├── send_local_plan.sh
│   │   │   └── source_llm_env.sh
│   │   ├── test/
│   │   │   └── test_task_validator.py
│   │   └── ur3_llm_control/
│   │       ├── camera_detector.py
│   │       ├── llm_planner.py
│   │       └── task_validator.py
│   └── ur3_llm_robot_skills/
│       ├── config/
│       │   ├── controllers.yaml
│       │   └── scene.yaml
│       ├── launch/
│       │   └── llm_robot.launch.py
│       ├── src/
│       │   └── robot_skills.cpp
│       ├── urdf/
│       │   ├── ur3_parallel_gripper.urdf.xacro
│       │   └── ur3_parallel_gripper.srdf.xacro
│       └── worlds/
│           └── ur3_pick_place.sdf
├── build/
├── install/
├── log/
└── README.md
```

## 4. Yêu cầu trước khi chạy

- Ubuntu 22.04 với ROS 2 Humble hoặc Ubuntu 24.04 với ROS 2 Jazzy.
- ROS packages MoveIt, UR, ROS 2 control, Gazebo bridge và OpenCV bridge.
- Node.js và 9Router đang chạy trên cổng `20128`.
- API key hợp lệ trong file cấu hình local.
- Công cụ `colcon`, `ros2` và `gazebo` đã được cài đặt.

## 5. Cấu hình API và sinh viên

Tạo file `src/ur3_llm_control/config/9router.env` nếu chưa có:

```env
LLM_ENDPOINT=http://localhost:20128/v1/chat/completions
LLM_MODEL=gemini/gemini-3.5-flash-lite
LLM_API_KEY=YOUR_9ROUTER_API_KEY
```

> Không đưa API key vào Git hoặc notebook. File này bị bỏ qua Git.

Cấu hình sinh viên hiện tại nằm trong `src/ur3_llm_control/config/student_config.yaml`:

- Học sinh: Nguyen Minh Nghia
- MSSV: 23020755
- Vật thể: `red_cube`, `yellow_cube`, `blue_cube`, `green_cube`, `purple_cube`
- Vùng: `zone_a`, `zone_b`, `zone_c`
- Vị trí tạm: `temp_1`, `temp_2`

In [ ]:
from pathlib import Path

workspace = Path.home() / "ur3_ws"
config = workspace / "src" / "ur3_llm_control" / "config" / "9router.env"
print(f"Workspace: {workspace}")
print(f"9Router config exists: {config.exists()}")
if config.exists():
    values = {}
    for line in config.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            values[key] = value
    print(f"LLM endpoint: {values.get('LLM_ENDPOINT', '<missing>')}")
    print(f"LLM model: {values.get('LLM_MODEL', '<missing>')}")
    print(f"API key configured: {'yes' if values.get('LLM_API_KEY') else 'no'}")

## 6. Build workspace

```bash
cd ~/ur3_ws
source /opt/ros/jazzy/setup.bash
colcon build --base-paths src --packages-up-to ur3_llm_control ur3_llm_robot_skills --symlink-install
source install/setup.bash
```

Nếu đang dùng ROS 2 Humble, thay `jazzy` bằng `humble`.

In [ ]:
import os
import shutil

print("ROS_DISTRO:", os.environ.get("ROS_DISTRO", "<not set>"))
print("colcon:", shutil.which("colcon"))
print("ros2:", shutil.which("ros2"))
print("gazebo:", shutil.which("gz"))

## 7. Khởi động hệ thống

```bash
cd ~/ur3_ws
src/ur3_llm_control/scripts/run_llm_robot.sh
```

Script này sẽ:

1. Kiểm tra 9Router configuration.
2. Khởi động 9Router trên cổng `20128` nếu chưa chạy.
3. Khởi động Gazebo, MoveIt 2 và các node của workspace.
4. Chờ gripper controller sẵn sàng trước khi executor start.

## 8. Demo câu lệnh

### Demo xử lý xung đột vùng

```bash
src/ur3_llm_control/scripts/send_demo_conflict.sh
```

Câu lệnh mẫu:

```text
Put the red cube in zone B
```

### Demo sắp xếp theo MSSV

```bash
src/ur3_llm_control/scripts/send_demo_advanced.sh
```

Câu lệnh mẫu:

```text
Arrange all objects according to my student ID
```

### Demo kế hoạch cục bộ

```bash
src/ur3_llm_control/scripts/send_local_plan.sh
```

Script này gửi kế hoạch trực tiếp lên topic `llm_task/validated_plan` và không gọi 9Router.

## 9. ROS 2 topics

| Topic | Mục đích |
|---|---|
| `/overhead_cam` | Ảnh camera Gazebo |
| `/scene_state` | Trạng thái vật thể và zone |
| `/llm_task/command` | Câu lệnh tự nhiên |
| `/llm_task/validated_plan` | Kế hoạch đã được validator kiểm tra |
| `/llm_task/execution_status` | Kết quả thực thi |

In [ ]:
import subprocess
import textwrap

commands = [
    ["ros2", "node", "list"],
    ["ros2", "topic", "list"],
]

for command in commands:
    print(f"\n$ {' '.join(command)}")
    result = subprocess.run(command, text=True, capture_output=True)
    print(result.stdout.strip() or result.stderr.strip())
    if result.returncode != 0:
        print(f"Return code: {result.returncode}")

## 10. Kiểm tra unit test

```bash
python3 -m unittest src/ur3_llm_control/test/test_task_validator.py
```

Hệ thống phải trả về `TASK SUCCESS` khi executor hoàn tất kế hoạch.

In [ ]:
import subprocess

command = ["python3", "-m", "unittest", "src/ur3_llm_control/test/test_task_validator.py"]
result = subprocess.run(command, text=True, capture_output=True)
print(result.stdout, end="")
print(result.stderr, end="")
print(f"Return code: {result.returncode}")

## 11. Các lỗi thường gặp

### 9Router config missing

Tạo file `src/ur3_llm_control/config/9router.env` và đặt API key hợp lệ.

### Port 20128 đã được dùng

Kiểm tra:

```bash
ss -tlnp | grep 20128
```

### ROS 2 environment chưa được load

```bash
source /opt/ros/jazzy/setup.bash
source install/setup.bash
```

### Node chưa chạy

```bash
ros2 node list
ros2 topic list
```

### Executor chưa nhận kế hoạch

Kiểm tra topic:

```bash
ros2 topic echo --no-arr /llm_task/execution_status
ros2 topic echo --no-arr /llm_task/validated_plan
```

## 12. Summary

- LLM chỉ chịu trách nhiệm lập kế hoạch.
- Validator kiểm tra kế hoạch trước khi execution.
- Camera và scene state cho planner biết trạng thái thực tế.
- MoveIt 2 điều khiển UR3 qua Gazebo.
- `home()` là bước kết thúc an toàn.
- API key và build artifacts không được lưu trong Git.